<a href="https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Ready.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #2 — "The Content Performance Curve" (health score by content age, showing a decay cliff at 271-365 days).

Label source: Health Score, a FlyRank composite (impressions 30pts + position 30pts + CTR 20pts + scroll depth 20pts) — not a single observed outcome like clicks or impressions, but a weighted blend. The paper's own methodology section notes health score is heavily driven by average_position (correlation -0.6 in the ML appendix), so an age-vs-health chart may partly be re-describing an age-vs-position relationship rather than a pure aging effect.

Does the validation design carry the claim? This is a direct aggregate comparison (mean health score per age bucket), not a held-out or split-validated result — the paper explicitly treats ML pages as secondary and these portfolio charts as primary. No sample sizes per age bucket are shown, and no significance testing is reported (the paper states this directly: "No p-values or confidence intervals are reported"). Without bucket sizes, it's hard to know if the 271-365 day "decay cliff" (health 14) is a robust pattern or driven by a handful of pages, especially since Finding #4 in the same paper explicitly flags that a similarly-shaped stale bucket (361+) was unstable due to a tiny sample (283:1 ratio from just 1 declining page). The paper is careful about this in one place but doesn't apply the same scrutiny to the age-curve buckets.

Finding #4 — "The Freshness Multiplier" (growth-to-decline ratio by freshness window, with a 31-90 day window as the strongest stable band).

Label source: growth-to-decline ratio, computed from trend_direction-style up/down labels within freshness buckets, on the local active-content sample.

Does the validation design carry the claim? The paper is unusually transparent about a limitation here — it explicitly discounts its own 361+ day bucket (283:1 ratio) as unstable due to tiny sample size (1 declining page), which is good practice I should adopt in my own work. But this raises a direct tension with my own w04_signal_audit finding: I tested whether days_since_last_update >= 180 (staleness) associates with higher decline rate on the FlyRank starter CSV, and found the opposite — stale pages declined less often (0.471) than fresh pages (0.542), a weak correlation (0.081). The paper's Finding #4 and Finding #8 both suggest freshness matters substantially. This could mean my dataset (a 30K-row anonymized sample) and the paper's dataset (341K pieces across 57 brands) behave differently, or it could mean one of the two staleness definitions is capturing something the other isn't. Rather than assume either result is "wrong," the honest takeaway is that the freshness-decline relationship is not settled by either analysis alone, and a reader should not treat "refresh stale content" as a guaranteed lever without checking it against their own portfolio.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 supporting check — reproduce my own staleness finding for direct comparison
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

stale_decline_rate = df[df["days_since_last_update"] >= 180]["is_declining_label"].mean()
fresh_decline_rate = df[df["days_since_last_update"] < 180]["is_declining_label"].mean()

print("My w04_signal_audit result (30K anonymized sample):")
print(f"  Decline rate, stale pages (>=180d): {stale_decline_rate:.3f}")
print(f"  Decline rate, fresh pages (<180d):  {fresh_decline_rate:.3f}")
print(f"  Verdict: OPPOSITE of 'staleness predicts decline'")
print()
print("FlyRank paper Finding #4 (341K portfolio, growth-to-decline ratio by freshness):")
print("  31-90 day freshness window: 7.88:1 growth-to-decline ratio (strongest)")
print("  181-360 day freshness window: 3.13:1 (weaker but still growth-favoring)")
print("  -> Paper's data leans toward freshness mattering; mine did not confirm this on the starter sample.")

My w04_signal_audit result (30K anonymized sample):
  Decline rate, stale pages (>=180d): 0.471
  Decline rate, fresh pages (<180d):  0.542
  Verdict: OPPOSITE of 'staleness predicts decline'

FlyRank paper Finding #4 (341K portfolio, growth-to-decline ratio by freshness):
  31-90 day freshness window: 7.88:1 growth-to-decline ratio (strongest)
  181-360 day freshness window: 3.13:1 (weaker but still growth-favoring)
  -> Paper's data leans toward freshness mattering; mine did not confirm this on the starter sample.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

"Before" reproduces the in-sample-style comparison the way w02 first demonstrated it (no held-out clients). "After" is the actual grouped client-holdout split from w05_model, which is the honest number. Comparing the two shows how much a naive evaluation can overstate performance.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

feature_columns = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

model_df = df[["client_id", "is_declining_label"] + feature_columns].copy()
model_df[feature_columns] = model_df[feature_columns].replace([np.inf, -np.inf], np.nan)
for col in feature_columns:
    model_df[col] = model_df[col].fillna(model_df[col].median())

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

# "Before": naive in-sample fit/score on ALL data, no holdout
X_all, y_all = model_df[feature_columns], model_df["is_declining_label"]
naive_model = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000, random_state=42))])
naive_model.fit(X_all, y_all)
naive_scores = naive_model.predict_proba(X_all)[:, 1]
naive_p50 = precision_at_k(y_all, naive_scores, 50)

# "After": honest grouped client-holdout split (same as w05_model)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(model_df, groups=model_df["client_id"]))
train, test = model_df.iloc[train_idx], model_df.iloc[test_idx]
X_train, y_train = train[feature_columns], train["is_declining_label"]
X_test, y_test = test[feature_columns], test["is_declining_label"]

honest_model = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000, random_state=42))])
honest_model.fit(X_train, y_train)
honest_scores = honest_model.predict_proba(X_test)[:, 1]
honest_p50 = precision_at_k(y_test, honest_scores, 50)

print(f"Before (naive, in-sample) Precision@50: {naive_p50:.3f}")
print(f"After  (honest, client-holdout) Precision@50: {honest_p50:.3f}")
print(f"Gap: {naive_p50 - honest_p50:.3f}")

Before (naive, in-sample) Precision@50: 0.820
After  (honest, client-holdout) Precision@50: 0.720
Gap: 0.100


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same hunt as w03_feature_leakage_check, run again on the final feature set actually used in w05_model — confirming trend_pct/trend_direction never entered X, and that every feature's correlation with trend_pct stays low.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_features = ["trend_pct", "trend_direction", "content_id", "client_id",
                      "impression_tier", "position_tier"]
leaked = [c for c in excluded_features if c in feature_columns]
print("Leaked columns found in final feature set:", leaked if leaked else "None — clean.")

corrs = df[feature_columns].corrwith(df["trend_pct"]).sort_values(key=abs, ascending=False)
print("\nCorrelation of final feature set with trend_pct (label source):")
print(corrs.round(3))
print(f"\nMax absolute correlation: {corrs.abs().max():.3f}")

Leaked columns found in final feature set: None — clean.

Correlation of final feature set with trend_pct (label source):
avg_position              0.047
days_with_impressions    -0.032
impressions_90d           0.024
competition               0.014
days_since_last_update   -0.014
word_count               -0.009
engagement_rate           0.008
ctr                       0.008
days_with_sessions       -0.008
char_count               -0.007
scroll_events_90d        -0.005
cpc                       0.005
ai_traffic_pct           -0.004
users_90d                -0.004
ai_sessions_90d          -0.003
sessions_90d             -0.003
search_volume             0.003
scroll_rate               0.002
clicks_90d               -0.002
engaged_sessions_90d      0.002
pageviews_90d            -0.001
content_age_days          0.001
dtype: float64

Max absolute correlation: 0.047


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Boldest original sentence (from w05_model, Section 4): "The strongest learned model was Logistic Regression, with Precision@50 = 0.720."

Rewritten in safe language: On this 30,000-page anonymized sample, using a client-holdout split, a logistic regression model achieved an observed Precision@50 of 0.720 — meaning roughly 36 of the top 50 ranked pages were labeled as declining in this dataset. This is a decision-support ranking signal for review prioritization, not a causal claim that these specific pages will decline, nor a guarantee of similar performance on unseen clients or future data.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Original claim: 'The strongest learned model was Logistic Regression, with Precision@50 = 0.720.'")
print("\nRewritten claim:")
print("On this 30,000-page anonymized sample, using a client-holdout split, a logistic")
print("regression model achieved an observed Precision@50 of 0.720 (~36 of top 50 pages")
print("labeled declining). This is decision-support, not causal, and not guaranteed to")
print("generalize beyond this sample.")

Original claim: 'The strongest learned model was Logistic Regression, with Precision@50 = 0.720.'

Rewritten claim:
On this 30,000-page anonymized sample, using a client-holdout split, a logistic
regression model achieved an observed Precision@50 of 0.720 (~36 of top 50 pages
labeled declining). This is decision-support, not causal, and not guaranteed to
generalize beyond this sample.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.